# CUBIC MARS -- PS1 Failure Prediction -- **GATE** (RVG / HBG / SAG)

**Problem statement (PS1):** predict whether a fare device will suffer a *chargeable* SLA
failure within the next **3 days** (`will_fail_3d`), at **(DEVICE_ID, transit_day)** grain.
Champion target = 3-day; 7-day and 14-day are scored as secondary lead-time context.

**Scope (this notebook):** `mars_device_category = 'GATE'` only (RVG / HBG / SAG; legacy
turnstiles TT/TTC/TWA/TEX already dropped upstream). Separate notebooks cover TVM and VALIDATOR.

**Label source (GATE):** `silver.device_outage` chargeable outages
(`is_chargeable = TRUE AND failure_level IN (2,3,4,5)`), rolled forward 1-3 days in
`gold.device_ps1_daily`. GATE fleet = ~1,382 current devices; 944 have >=1 failure in window.

**Promotion gates (v7.0 -- never raw accuracy at this base rate):**
Average Precision / PR-AUC (primary) + ROC-AUC + **recall floor 0.70** at the operating
threshold + precision@K. A leakage-scan gate (top solo-feature AUC < 0.95) must pass.

**What this notebook delivers:**
1. Robust load (direct Databricks) + data-quality profile of the GATE slice.
2. Leakage-safe preprocessing + device-grouped, past-only feature engineering.
3. Automated leakage scan + near-zero-variance / all-null pruning.
4. Time-aware **purged** cross-validation with an embargo gap (no future/label leakage).
5. Multi-model bake-off: LogReg, RandomForest, HistGBM, XGBoost, LightGBM, CatBoost,
   a weighted soft-vote ensemble (RF 0.45 / XGB 0.35 / Cat 0.20) and a stacked ensemble.
6. Optuna fine-tuning on **cross-validated AP** (tuned on CV folds only -- test never touched;
   fixes the val-set-leakage over-fit seen in the Jul-13 run).
7. Probability calibration (Platt/sigmoid -- isotonic collapsed GATE AUC before) + operating-
   threshold selection to meet the recall floor.
8. SHAP explainability + per-fold stability.
9. **Scoring against each DEVICE_ID** + a **serial / component attribution table** joined via
   `silver.hw_config_current` (one row per component serial), for work-order routing.
10. MLflow logging + optional Model Package Group registration + serving-package hooks.

> Grounded in `sql/gold/device_ps1_daily__create.sql` (107 cols) + the Chicago data catalog
> (snapshot 2026-07-17). Every feature here is same-day-or-past: the label only looks *forward*
> 1-3 days, so there is **no future leakage by construction** -- the guards below defend against
> *concurrency* (a device already failing today) and hyper-parameter/test leakage.


### Dependencies (Databricks single-user cluster, e.g. `mars-dev-interactive`)

If any of xgboost / lightgbm / catboost / optuna / shap / sagemaker-mlflow are missing, install
them **without upgrading numpy** (numpy 2.x breaks the Studio/DBR kernel -- the hard-won 2026-07 lesson),
then restart Python:

**On a Databricks single-user cluster** (`%pip` works there, not on shared USER_ISOLATION clusters):
```
%pip install --quiet "numpy<2" xgboost lightgbm catboost optuna shap sagemaker-mlflow
dbutils.library.restartPython()
```

**In SageMaker Studio** (the default S3-export path) you also need `pyarrow` + `s3fs` to read the
gold/silver parquet exports from S3:
```
%pip install --quiet "numpy<2" pyarrow s3fs xgboost lightgbm catboost optuna shap sagemaker-mlflow
```

Every optional library is import-guarded below, so the notebook still runs (smaller model roster /
no MLflow) if some are missing -- but `pyarrow`+`s3fs` are required for the S3 read.

In [ ]:
# =========================== CONFIG (edit here) =============================
# The only block that changes between the GATE / TVM / VALIDATOR notebooks.
CONFIG = {
    # ---- scope ----
    "DEVICE_CATEGORY":   "GATE",              # 'GATE' | 'TVM' | 'VALIDATOR'
    "PRIMARY_HORIZON":   "will_fail_3d",       # champion target
    "SECONDARY_HORIZONS": ["will_fail_7d", "will_fail_14d"],
    "RECALL_FLOOR":      0.70,                  # v7.0 GATE operating-threshold recall floor (TVM=0.80)

    # ---- data source ----
    # 'auto' tries an active Spark session (Databricks cluster) -> databricks-sql -> S3 parquet.
    # Default below = the S3 gold/silver export, the SageMaker Studio path (no Spark needed).
    # First run notebooks/export_gold_to_s3.py + export_silver_to_s3.py in Databricks.
    "DATA_SOURCE":       "auto",               # 'databricks_spark' | 'databricks_sql' | 'parquet' | 'auto'
    "GOLD_TABLE":        "mars_dev.gold.device_ps1_daily",
    "HWCONFIG_TABLE":    "mars_dev.silver.hw_config_current",
    # databricks-sql fallback (only used if no Spark session; leave token in an env var)
    "DBX_HOST":          "",                   # e.g. dbc-c05ec222-36c5.cloud.databricks.com
    "DBX_HTTP_PATH":     "",                   # e.g. /sql/1.0/warehouses/xxxxxxxx
    "DBX_TOKEN_ENV":     "DATABRICKS_TOKEN",   # env var holding the PAT
    # S3 parquet export (DEFAULT for SageMaker Studio; both gold + silver land in the gold bucket)
    "GOLD_PARQUET":      "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/gold/device_ps1_daily",
    "HWCONFIG_PARQUET":  "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/silver/hw_config_current",
    "S3_REGION":         "us-east-1",          # region for pd.read_parquet storage_options on s3://

    # ---- optional in-notebook feature-source joins (leakage-safe prior-day windows) ----
    # These add signal NOT in device_ps1_daily. Toggle off to fall back to gold-only features.
    "ENRICH_SOURCES":       True,
    "ENRICH_USAGE":         True,   # silver.usage_lifecycle_daily -- maintenance activity + cumulative wear
    "ENRICH_STATION":       True,   # silver.station_network_daily -- station co-failure stress (prior-day)
    "ENRICH_READTAP":       True,   # silver.read_tap_daily        -- reader reject-rate degradation
    "ENRICH_COMPONENT_AGE": True,   # silver.hw_config_current     -- device-level component-age aggregates
    "USAGE_TABLE":     "mars_dev.silver.usage_lifecycle_daily",
    "USAGE_PARQUET":   "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/silver/usage_lifecycle_daily",
    "STATION_TABLE":   "mars_dev.silver.station_network_daily",
    "STATION_PARQUET": "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/silver/station_network_daily",
    "READTAP_TABLE":   "mars_dev.silver.read_tap_daily",
    "READTAP_PARQUET": "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/silver/read_tap_daily",

    # ---- NEW: degradation-warning precursors (biggest AUC lever) ----
    # Prereq: run WARNING_EXPORT_SQL section in export_silver_to_s3.py in Databricks first.
    "ENRICH_WARNINGS":      True,   # silver.warnings_daily -- precursor events before OOS failure
    "WARNINGS_TABLE":       "mars_dev.silver.warnings_daily",
    "WARNINGS_PARQUET":     "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/silver/warnings_daily",

    # ---- NEW: heartbeat / silence signal (Tier-1 PdM signal) ----
    "ENRICH_UPTIME":        True,   # silver.device_uptime_intervals -- device silence precedes failure
    "UPTIME_TABLE":         "mars_dev.silver.device_uptime_intervals",
    "UPTIME_PARQUET":       "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/silver/device_uptime_intervals",

    # ---- temporal split + CV ----
    "TEST_FRACTION":     0.20,                 # most-recent share of dates held out for final test
    "EMBARGO_DAYS":      14,                    # purge gap = max label horizon (14d) to stop label bleed
    "CV_SPLITS":         4,                     # purged expanding-window folds for tuning/stability
    "MIN_TRAIN_DATE":    "2024-01-01",

    # ---- feature policy ----
    # Same-day "the device is failing right now" columns. Excluded from the champion by default
    # (PS1 is preventive -> predict BEFORE the outage). An ablation re-adds them so you see the lift.
    "USE_CONCURRENT_FEATURES": False,
    "CONCURRENT_FEATURES": [
        "outage_count", "total_outage_min", "max_outage_min",
        "chargeable_outage_count", "chargeable_outage_min", "max_failure_level",
        "oos_event_count", "hardware_oos_count",
    ],
    "LEAKAGE_AUC_THRESHOLD": 0.95,             # drop any single feature whose solo-AUC exceeds this
    "NZV_UNIQUE_MIN":    2,                     # drop columns with < 2 distinct values on train

    # ---- tuning ----
    "RUN_OPTUNA":        True,
    "N_OPTUNA_TRIALS":   30,                    # raise for a deeper search (slower)
    "TUNE_SAMPLE_ROWS":  400_000,              # cap tuning rows for speed (None = use all)
    "RANDOM_STATE":      42,

    # ---- calibration ----
    "CALIBRATION_METHOD": "sigmoid",           # 'sigmoid' (Platt) -- isotonic collapsed GATE AUC before
    "CALIB_FRACTION":    0.15,                  # temporal tail of TRAIN used to calibrate

    # ---- scoring / precision@K ----
    "PRECISION_AT_K":    [0.01, 0.05, 0.10],   # top-1/5/10% of scored rows
    "RISK_BANDS":        [(0.0, 0.10, "LOW"), (0.10, 0.30, "MEDIUM"),
                          (0.30, 0.60, "HIGH"), (0.60, 1.01, "CRITICAL")],

    # ---- MLflow / registry (optional; safe no-ops if left blank) ----
    "MLFLOW_TRACKING_URI": "",                 # SageMaker-managed MLflow ARN, e.g. arn:aws:sagemaker:us-east-1:170202974600:mlflow-tracking-server/cubic-mars-mlflow-server-dev
    "MLFLOW_EXPERIMENT":   "chicago-ps1-gate-failure",
    "REGISTER_MODEL":      False,
    "MODEL_PACKAGE_GROUP": "cubic-mars-models-gate-ps1-dev",
    "AWS_REGION":          "us-east-1",

    # ---- outputs ----
    "OUT_DIR":           "PS1_outputs_pavan/gates",   # per-device subfolder under PS1_outputs_pavan
    "WRITE_SCORES_TO_DBX": False,              # if True + Spark present, write scored tables back to UC
    "SCORES_TABLE":      "mars_dev.gold.ps1_gate_device_scores",
    "ATTRIB_TABLE":      "mars_dev.gold.ps1_gate_component_attribution",
}
CATEGORY = CONFIG["DEVICE_CATEGORY"]
print("PS1 notebook configured for:", CATEGORY, "| primary target:", CONFIG["PRIMARY_HORIZON"])


In [ ]:
# =========================== Environment & imports =========================
import os, sys, json, math, warnings, time, textwrap
os.environ.setdefault("AWS_REGION",         CONFIG["AWS_REGION"])   # Jul-13 bug: undefined region skipped SM registration
os.environ.setdefault("AWS_DEFAULT_REGION", CONFIG["AWS_REGION"])
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

# Optional deps degrade gracefully (guarded everywhere they are used).
def _try(name):
    try:
        return __import__(name)
    except Exception as e:
        print(f"[optional] {name} unavailable ({type(e).__name__}); paths using it will be skipped")
        return None
xgb   = _try("xgboost")
lgb   = _try("lightgbm")
catb  = _try("catboost")
optuna_mod = _try("optuna")
shap_mod   = _try("shap")

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, StackingClassifier, VotingClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (average_precision_score, roc_auc_score, brier_score_loss,
                             precision_score, recall_score, f1_score, fbeta_score,
                             precision_recall_curve, confusion_matrix)

OUT = Path(CONFIG["OUT_DIR"]); OUT.mkdir(parents=True, exist_ok=True)
# ---- capture every cell's console output to a run log inside OUT (tees stdout + stderr) ----
class _Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, d):
        for s in self.streams:
            try: s.write(d); s.flush()
            except Exception: pass
    def flush(self):
        for s in self.streams:
            try: s.flush()
            except Exception: pass
if "_ORIG_STDOUT" not in globals():
    _ORIG_STDOUT, _ORIG_STDERR = sys.stdout, sys.stderr     # real notebook streams (display still works)
_run_log = open(OUT / "run_console_log.txt", "w", encoding="utf-8")   # fresh log per Run All
sys.stdout = _Tee(_ORIG_STDOUT, _run_log); sys.stderr = _Tee(_ORIG_STDERR, _run_log)
print(f"[log] capturing all cell output to {OUT/'run_console_log.txt'}")
RS  = CONFIG["RANDOM_STATE"]
np.random.seed(RS)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 60)
print("Imports OK. Output dir:", OUT.resolve())
print("Boosters available -> xgboost:", xgb is not None, "| lightgbm:", lgb is not None, "| catboost:", catb is not None)


In [ ]:
# =========================== Data loading (direct Databricks) ==============
# Robust loader: active Spark session first (Databricks), then databricks-sql, then parquet.
def _active_spark():
    g = globals().get("spark", None)
    if g is not None:
        return g
    try:
        from pyspark.sql import SparkSession
        return SparkSession.getActiveSession()
    except Exception:
        return None

def _load_via_sql(table, where=None):
    from databricks import sql as dbsql
    tok = os.environ.get(CONFIG["DBX_TOKEN_ENV"], "")
    assert CONFIG["DBX_HOST"] and CONFIG["DBX_HTTP_PATH"] and tok, (
        "databricks-sql fallback needs DBX_HOST, DBX_HTTP_PATH and the token env var")
    q = f"SELECT * FROM {table}" + (f" WHERE {where}" if where else "")
    with dbsql.connect(server_hostname=CONFIG["DBX_HOST"], http_path=CONFIG["DBX_HTTP_PATH"],
                       access_token=tok) as conn:
        with conn.cursor() as cur:
            cur.execute(q)
            try:
                return cur.fetchall_arrow().to_pandas()
            except Exception:
                rows = cur.fetchall(); cols = [d[0] for d in cur.description]
                return pd.DataFrame(rows, columns=cols)

def load_table(table, parquet_path, category_filter=None):
    src = CONFIG["DATA_SOURCE"]
    sp = _active_spark()
    where = f"mars_device_category = '{category_filter}'" if category_filter else None
    if src in ("auto", "databricks_spark") and sp is not None:
        sdf = sp.table(table)
        if category_filter:
            sdf = sdf.filter(f"mars_device_category = '{category_filter}'")
        print(f"[load] Spark -> {table}" + (f" [{category_filter}]" if category_filter else ""))
        return sdf.toPandas()
    if src in ("auto", "databricks_sql") and (CONFIG["DBX_HOST"] and CONFIG["DBX_HTTP_PATH"]):
        print(f"[load] databricks-sql -> {table}")
        return _load_via_sql(table, where)
    if src in ("auto", "parquet") and parquet_path:
        print(f"[load] parquet -> {parquet_path}")
        rk = {}
        if str(parquet_path).startswith("s3"):
            rk["storage_options"] = {"client_kwargs": {"region_name": CONFIG.get("S3_REGION", "us-east-1")}}
        try:
            if category_filter:
                # pyarrow predicate pushdown: read only this category's row groups (avoids
                # pulling all ~2.9M device-days into memory just to filter to one device type)
                return pd.read_parquet(parquet_path,
                                       filters=[("mars_device_category", "==", category_filter)], **rk)
            return pd.read_parquet(parquet_path, **rk)
        except (ImportError, ModuleNotFoundError) as e:
            raise RuntimeError(f"reading {parquet_path} needs pyarrow + s3fs -> "
                               f"%pip install pyarrow s3fs   (underlying: {e})")
        except Exception as e:
            print(f"[load] pushdown/read fallback ({type(e).__name__}: {e}); full read then filter")
            d = pd.read_parquet(parquet_path, **rk)
            if category_filter and "mars_device_category" in d.columns:
                d = d[d["mars_device_category"] == category_filter].copy()
            return d
    raise RuntimeError(
        "No usable data source. Checked -> Spark session: "
        + ("present" if sp is not None else "MISSING (not on a Databricks cluster)")
        + f" | databricks-sql params set: {bool(CONFIG['DBX_HOST'] and CONFIG['DBX_HTTP_PATH'])}"
        + f" | parquet path set: {bool(parquet_path)}.  Fix ONE of: "
        + "(a) run this notebook on a Databricks cluster (Spark auto-detected, zero config); "
        + "(b) DATA_SOURCE='databricks_sql' + DBX_HOST + DBX_HTTP_PATH + token env var; "
        + "(c) DATA_SOURCE='parquet'|'auto' + GOLD_PARQUET/HWCONFIG_PARQUET S3 export paths (the default).")

# ---- environment probe (prints exactly what the loader can see) ----
_probe_sp = _active_spark()
print("[env] Spark session:", ("present -> will read UC directly" if _probe_sp is not None
      else "MISSING -> not on a Databricks cluster; using databricks-sql or the S3 parquet export"))
print("[env] DATA_SOURCE:", CONFIG["DATA_SOURCE"], "| GOLD_PARQUET set:", bool(CONFIG["GOLD_PARQUET"]),
      "| DBX SQL params set:", bool(CONFIG["DBX_HOST"] and CONFIG["DBX_HTTP_PATH"]))

df = load_table(CONFIG["GOLD_TABLE"], CONFIG["GOLD_PARQUET"], category_filter=CATEGORY)
df["transit_day"] = pd.to_datetime(df["transit_day"])
df = df[df["transit_day"] >= pd.Timestamp(CONFIG["MIN_TRAIN_DATE"])].copy()
df = df.sort_values(["DEVICE_ID", "transit_day"]).reset_index(drop=True)
print(f"\nLoaded {len(df):,} {CATEGORY} device-days | {df['DEVICE_ID'].nunique():,} devices | "
      f"{df['transit_day'].min().date()} -> {df['transit_day'].max().date()}")


## 0b. External feature-source joins (in-notebook, leakage-safe)

Signal **not** in `device_ps1_daily`, joined here rather than in a new gold table. All windows are
strictly prior-day, each source is independently guarded (a missing S3 export just prints "skipped"),
and every new column flows through the leakage scan + NZV prune below:
* **`usage_lifecycle_daily`** (S20) -- cumulative wear (`days_in_service`, `cumulative_tap_count`,
  `tap_count_30d`) + maintenance (`days_since_last_maintenance`, `maint_events_30d`, and 7-day-prior
  rolling tech-logins / maint-events / maint-mode).
* **`station_network_daily`** (S27) -- prior-7-day station co-failure stress (devices failed,
  coordinated / major-station events) at the device's facility.
* **`read_tap_daily`** (S22) -- reader reject-rate + null-status rate (same-day + prior-7d).
* **`hw_config_current`** -- device-level component-age min/mean/max/count.

In [ ]:
# =========================== external feature-source joins ================
ENRICHED_COLS = []
def _num(s):  return pd.to_numeric(s, errors="coerce")      # Decimal/object -> float
def _proll(s, how, w=7):                                     # prior-day rolling (excludes today)
    r = s.shift(1).rolling(w, min_periods=1)
    return getattr(r, how)()

if CONFIG.get("ENRICH_SOURCES", True):
    # 1) usage_lifecycle_daily (S20): maintenance activity + cumulative wear -------------
    if CONFIG.get("ENRICH_USAGE", True):
        try:
            ul = load_table(CONFIG["USAGE_TABLE"], CONFIG["USAGE_PARQUET"], category_filter=CATEGORY)
            ul["transit_day"] = pd.to_datetime(ul["transit_day"])
            past = [c for c in ["days_in_service","cumulative_tap_count","tap_count_30d",
                                "cumulative_maint_events","days_since_last_maintenance","maint_events_30d"]
                    if c in ul.columns]
            same = [c for c in ["daily_maint_events","daily_tech_logins","daily_maint_mode_events"]
                    if c in ul.columns]
            ul = ul[["DEVICE_ID","transit_day"] + past + same].drop_duplicates(["DEVICE_ID","transit_day"])
            for c in past + same: ul[c] = _num(ul[c])
            df = df.merge(ul, on=["DEVICE_ID","transit_day"], how="left").sort_values(["DEVICE_ID","transit_day"])
            for c in past:
                df.rename(columns={c: f"ul_{c}"}, inplace=True); ENRICHED_COLS.append(f"ul_{c}")
            for c in same:                                   # same-day activity -> prior-7d sum
                nn = f"ul_{c}_7d_prior"
                df[nn] = df.groupby("DEVICE_ID")[c].transform(lambda s: _proll(s, "sum"))
                df.drop(columns=[c], inplace=True); ENRICHED_COLS.append(nn)
            print(f"[enrich] usage_lifecycle_daily -> +{len(past)+len(same)} cols")
        except Exception as e:
            print("[enrich] usage_lifecycle skipped:", type(e).__name__, e)

    # 2) station_network_daily (S27): prior-day station co-failure stress ----------------
    if CONFIG.get("ENRICH_STATION", True):
        try:
            st = load_table(CONFIG["STATION_TABLE"], CONFIG["STATION_PARQUET"])
            cc = next((c for c in ["device_category","mars_device_category"] if c in st.columns), None)
            if cc: st = st[st[cc] == CATEGORY]
            st["transit_day"] = pd.to_datetime(st["transit_day"])
            for c in ["devices_failed","total_failure_events","is_coordinated_failure","is_major_station_event"]:
                if c in st.columns: st[c] = _num(st[c])
            st = st.groupby(["FACILITY_ID","transit_day"], as_index=False).agg(
                    stn_devices_failed=("devices_failed","sum"),
                    stn_failure_events=("total_failure_events","sum"),
                    stn_coordinated=("is_coordinated_failure","max"),
                    stn_major=("is_major_station_event","max"))
            st["_facnum"] = _num(st["FACILITY_ID"]); df["_facnum"] = _num(df["FACILITY_ID"])
            df = df.merge(st[["_facnum","transit_day","stn_devices_failed","stn_failure_events",
                              "stn_coordinated","stn_major"]], on=["_facnum","transit_day"], how="left")
            df.drop(columns=["_facnum"], inplace=True)
            df = df.sort_values(["DEVICE_ID","transit_day"])
            for c, how in [("stn_devices_failed","sum"),("stn_failure_events","sum"),
                           ("stn_coordinated","max"),("stn_major","max")]:
                df[c] = df[c].fillna(0.0)
                nn = f"{c}_prior7"
                df[nn] = df.groupby("DEVICE_ID")[c].transform(lambda s, h=how: _proll(s, h))
                df.drop(columns=[c], inplace=True); ENRICHED_COLS.append(nn)
            print("[enrich] station_network_daily -> +4 prior-day cols")
        except Exception as e:
            print("[enrich] station_network skipped:", type(e).__name__, e)

    # 3) read_tap_daily (S22): reader reject / null-status degradation --------------------
    if CONFIG.get("ENRICH_READTAP", True):
        try:
            rt = load_table(CONFIG["READTAP_TABLE"], CONFIG["READTAP_PARQUET"], category_filter=CATEGORY)
            rt["transit_day"] = pd.to_datetime(rt["transit_day"])
            for c in ["daily_read_count","rejected_read_count","null_status_read_count"]:
                if c in rt.columns: rt[c] = _num(rt[c])
            rt = rt.groupby(["DEVICE_ID","transit_day"], as_index=False).agg(
                    reads=("daily_read_count","sum"), rej=("rejected_read_count","sum"),
                    nul=("null_status_read_count","sum"))
            denom = rt["reads"].where(rt["reads"] > 0)
            rt["rt_reject_rate"] = 100.0 * rt["rej"] / denom
            rt["rt_null_rate"]   = 100.0 * rt["nul"] / denom
            df = df.merge(rt[["DEVICE_ID","transit_day","rt_reject_rate","rt_null_rate"]],
                          on=["DEVICE_ID","transit_day"], how="left").sort_values(["DEVICE_ID","transit_day"])
            for c in ["rt_reject_rate","rt_null_rate"]:
                df[f"{c}_prior7"] = df.groupby("DEVICE_ID")[c].transform(lambda s: _proll(s, "mean"))
                ENRICHED_COLS += [c, f"{c}_prior7"]
            print("[enrich] read_tap_daily -> +4 cols (same-day + prior-7d reject/null rate)")
        except Exception as e:
            print("[enrich] read_tap skipped:", type(e).__name__, e)

    # 4) component-age aggregates from hw_config_current (leakage-free) -------------------
    if CONFIG.get("ENRICH_COMPONENT_AGE", True):
        try:
            hwc = load_table(CONFIG["HWCONFIG_TABLE"], CONFIG["HWCONFIG_PARQUET"], category_filter=CATEGORY)
            hwc["component_age_days"] = _num(hwc["component_age_days"])
            agg = hwc.groupby("DEVICE_ID").agg(
                    cmp_age_min=("component_age_days","min"), cmp_age_mean=("component_age_days","mean"),
                    cmp_age_max=("component_age_days","max"),
                    cmp_count=("COMPONENT_SERIAL_NBR","nunique")).reset_index()
            df = df.merge(agg, on="DEVICE_ID", how="left")
            ENRICHED_COLS += ["cmp_age_min","cmp_age_mean","cmp_age_max","cmp_count"]
            print("[enrich] hw_config component age -> +4 cols")
        except Exception as e:
            print("[enrich] component_age skipped:", type(e).__name__, e)

# 5) device_uptime_intervals (S05): heartbeat / silence -- a device going quiet precedes failure
    if CONFIG.get("ENRICH_UPTIME", True):
        try:
            up = load_table(CONFIG.get("UPTIME_TABLE", ""), CONFIG.get("UPTIME_PARQUET", ""), category_filter=CATEGORY)
            up["transit_day"] = pd.to_datetime(up["transit_day"])
            ut_cols = [c for c in ["uptime_pct","downtime_hours","hours_since_last_hb",
                                   "is_silent_device","distinct_message_types","total_msg_count"]
                       if c in up.columns]
            up = up[["DEVICE_ID","transit_day"] + ut_cols].drop_duplicates(["DEVICE_ID","transit_day"])
            for c in ut_cols: up[c] = _num(up[c])
            df = df.merge(up, on=["DEVICE_ID","transit_day"], how="left").sort_values(["DEVICE_ID","transit_day"])
            # is_silent_device: today's value is safe (it's a current state indicator, not a future one)
            # For uptime_pct / downtime_hours we also want a 7d trailing average
            for c in ut_cols:
                df[c] = df[c].fillna(0.0 if c != "uptime_pct" else 100.0)
            for c in ["uptime_pct", "downtime_hours"]:
                if c in ut_cols:
                    nn = f"{c}_prior7"
                    df[nn] = df.groupby("DEVICE_ID")[c].transform(lambda s: _proll(s, "mean"))
                    ENRICHED_COLS.append(nn)
            ENRICHED_COLS += ut_cols
            print(f"[enrich] device_uptime_intervals -> +{len(ut_cols)+2} cols (uptime + silence + prior-7d)")
        except Exception as e:
            print("[enrich] device_uptime_intervals skipped:", type(e).__name__, e)

# 6) warnings_daily (computed): degradation-warning precursors -- biggest AUC lever
    #    Low / Near-Full / service-call events that fire *before* the OOS failure.
    #    Source: WARNING_EXPORT_SQL in warnings.py (aggregated from device_event_enriched).
    if CONFIG.get("ENRICH_WARNINGS", True):
        try:
            wd = load_table(CONFIG.get("WARNINGS_TABLE", ""), CONFIG.get("WARNINGS_PARQUET", ""),
                            category_filter=None)   # not device-category scoped at source
            wd["event_date"] = pd.to_datetime(wd["event_date"])
            wd = wd.rename(columns={"event_date": "transit_day"})
            WARN_RAW = ["warn_events","service_call_events","nearfull_events","low_events",
                        "warn_chu","warn_bhu","warn_printer","warn_comms","warn_reader"]
            for c in WARN_RAW:
                if c not in wd.columns: wd[c] = 0.0
                wd[c] = _num(wd[c]).fillna(0.0)
            wd = wd[["DEVICE_ID","transit_day"] + WARN_RAW].drop_duplicates(["DEVICE_ID","transit_day"])
            df = df.merge(wd, on=["DEVICE_ID","transit_day"], how="left").sort_values(["DEVICE_ID","transit_day"])
            for c in WARN_RAW:
                df[c] = df[c].fillna(0.0)

            # ---- recency (days since last warning / service call / near-full) ----
            _RCAP = 180
            def _days_since_w(flag_s):
                dt = df["transit_day"]
                last = dt.where(flag_s.to_numpy().astype(bool))
                last = last.groupby(df["DEVICE_ID"]).ffill()
                return (dt - last).dt.days.fillna(_RCAP).clip(0, _RCAP).astype("float32")

            df["w_days_since_warning"]      = _days_since_w(df["warn_events"]       > 0)
            df["w_days_since_service_call"] = _days_since_w(df["service_call_events"] > 0)
            df["w_days_since_nearfull"]     = _days_since_w(df["nearfull_events"]   > 0)

            # ---- trailing-window sums (7d / 30d) ----
            _WIN_W = (7, 30)
            for col in WARN_RAW:
                for w in _WIN_W:
                    nn = f"w_{col}_{w}d"
                    df[nn] = (df.groupby("DEVICE_ID")[col]
                                .transform(lambda x: x.rolling(w, min_periods=1).sum())
                                .astype("float32"))
                    ENRICHED_COLS.append(nn)

            # ---- acceleration: rising degradation (current 7d vs prior 7d) ----
            _prev7 = df.groupby("DEVICE_ID")["warn_events"].transform(
                lambda x: x.rolling(14, min_periods=1).sum() - x.rolling(7, min_periods=1).sum())
            df["w_warn_accel_7d"]   = (df["w_warn_events_7d"] - _prev7).astype("float32")
            df["w_warned_recently"] = (df["w_days_since_warning"] <= 7).astype("int8")

            _w_recency = ["w_days_since_warning","w_days_since_service_call",
                          "w_days_since_nearfull","w_warn_accel_7d","w_warned_recently"]
            ENRICHED_COLS += _w_recency
            # drop the raw per-day counts (rolled-up versions carry the signal)
            df.drop(columns=[c for c in WARN_RAW if c in df.columns], inplace=True, errors="ignore")
            n_added = len(_w_recency) + len(WARN_RAW) * len(_WIN_W)
            print(f"[enrich] warnings_daily -> +{n_added} warning features "
                  f"(recency, 7d/30d sums per subsystem, acceleration, binary flag)")
        except Exception as e:
            print("[enrich] warnings_daily skipped:", type(e).__name__, e)

df = df.sort_values(["DEVICE_ID","transit_day"]).reset_index(drop=True)
print(f"[enrich] total added candidate columns: {len(ENRICHED_COLS)} "
      f"(all pass through the leakage scan + NZV prune next)")


## 1. Data-quality profile & label balance

Before modelling: confirm the GATE slice, the per-horizon positive rate (GATE is the
rare-positive device -- expect ~3-5% at 3d), null coverage, and how the positive rate moves
over time (a drifting base rate changes the operating threshold).

In [ ]:
# =========================== DQ profile ====================================
label_cols = [CONFIG["PRIMARY_HORIZON"]] + CONFIG["SECONDARY_HORIZONS"]
for lc in label_cols:
    if lc in df.columns:
        pr = df[lc].mean()
        print(f"  {lc:14s} positives = {int(df[lc].sum()):>8,}  /  {len(df):>9,}  "
              f"= {pr*100:6.2f}%   (imbalance 1:{(1-pr)/max(pr,1e-9):.0f})")

# rows per device + span
dev = df.groupby("DEVICE_ID").agg(rows=("transit_day","size"),
                                  first=("transit_day","min"), last=("transit_day","max"))
print(f"\ndevices: {len(dev):,} | median rows/device: {dev['rows'].median():.0f} | "
      f"min {dev['rows'].min()} max {dev['rows'].max()}")

# monthly positive rate (base-rate drift)
mo = (df.assign(month=df["transit_day"].dt.to_period("M").astype(str))
        .groupby("month")[CONFIG["PRIMARY_HORIZON"]].agg(["mean","size"]))
print("\nmonthly positive rate (primary horizon):")
print((mo["mean"]*100).round(2).tail(12).to_string())

# null coverage (top 15 nullable feature columns)
nulls = df.isna().mean().sort_values(ascending=False)
nulls = nulls[nulls > 0].head(15)
if len(nulls):
    print("\ntop null-rate columns (kept as NaN -> imputed / native):")
    print((nulls*100).round(1).to_string())
else:
    print("\nno null columns.")


## 2. Column classification (leakage-safe)

Every gold column is same-day-or-past, so there is **no future leakage**. We still separate:
* **IDs / metadata** -- never features (device/facility/operator/serial/category/array keys).
* **Labels** -- the three forward windows.
* **Concurrent-failure signals** -- same-day outage/OOS columns (`chargeable_outage_*`,
  `max_failure_level`, `oos_event_count`, ...). These say *the device is failing right now*.
  PS1 is preventive, so they are **excluded from the champion by default** and re-tested in an
  ablation. Rolling 7/30-day windows are kept (history-dominated, not future).
* **Candidate features** -- everything else (telemetry, comms, tap-timing, KPI, revenue,
  past-only failure/incident history).

In [ ]:
# =========================== classify columns ==============================
ID_META = ["DEVICE_ID","transit_day","DEVICE_KEY","DEVICE_NAME","FACILITY_ID","FACILITY_NAME",
           "OPERATOR_ID","OPERATOR_NAME","DEVICE_SERIAL_NUMBER","mars_device_category",
           "DEVICE_CONTROL_GROUP_TYPE_NAME","BUS_ID","TRANSIT_MODE_NAME",
           "TRANSIT_ARRAY_ID","ARRAY_POSITION","FARE_CONTROL_AREA"]
LABELS   = ["will_fail_3d","will_fail_7d","will_fail_14d"]
CONCURRENT = CONFIG["CONCURRENT_FEATURES"]

# cast booleans (incl. pandas nullable 'boolean') -> float so every candidate is numeric + NaN-safe
for c in df.columns:
    if df[c].dtype == bool or str(df[c].dtype) == "boolean":
        df[c] = df[c].astype("float64")

# Coerce Spark DECIMAL columns to float64. The S3 parquet export reads DECIMAL back as python
# Decimal objects (object dtype); Decimal/float arithmetic in feature engineering raises
# "unsupported operand type(s) for /: 'decimal.Decimal' and 'float'", AND object-dtype columns
# would be silently dropped from features (they fail the is_numeric_dtype candidate check below).
_noncoerce = set(ID_META) | set(LABELS) | {"transit_day"}
_coerced = []
for c in df.columns:
    if c in _noncoerce or pd.api.types.is_numeric_dtype(df[c]):
        continue
    conv = pd.to_numeric(df[c], errors="coerce")          # Decimal/str -> float; text -> NaN
    if conv.notna().any() or df[c].isna().all():          # skip genuine string columns
        df[c] = conv.astype("float64"); _coerced.append(c)
if _coerced:
    print(f"coerced {len(_coerced)} DECIMAL/object cols -> float64: {_coerced[:15]}"
          + (" ..." if len(_coerced) > 15 else ""))

blocked = set(ID_META) | set(LABELS)
if not CONFIG["USE_CONCURRENT_FEATURES"]:
    blocked |= set(CONCURRENT)

candidate = []
for c in df.columns:
    if c in blocked:
        continue
    if pd.api.types.is_numeric_dtype(df[c]):
        candidate.append(c)

print(f"candidate raw features: {len(candidate)}")
print(f"concurrent-failure cols held out (ablation later): "
      f"{sorted(set(CONCURRENT) & set(df.columns)) if not CONFIG['USE_CONCURRENT_FEATURES'] else 'INCLUDED'}")


## 3. Feature engineering (device-grouped, strictly past)

All engineered features are computed **within each device's time series** and use `shift`/
rolling over *prior* days only, so nothing peeks forward. We add short-window momentum,
volatility, day-over-day deltas, trend ratios, device tenure, a comms x unavailability stress
proxy, and light calendar features (day-of-week / month / weekend -- quarter omitted to curb the
GATE calendar over-reliance seen on 2026-07-13).

On top of the enrichment joins we build **cross-source derived features** (each guarded on
availability): usage intensity (taps per service-day), usage acceleration (recent vs lifetime rate),
maintenance burden (maint events per service-day) and recent maintenance intensity, component-age
span, reader reject/null-rate **delta vs the 7-day baseline** (degradation slope), a station-stress
composite, and age x recent-failure / wear x rising-criticals interactions.

In [ ]:
# =========================== engineered features ===========================
def engineer_features(d, base_signals):
    d = d.sort_values(["DEVICE_ID","transit_day"]).copy()
    g = d.groupby("DEVICE_ID", group_keys=False)
    new = []
    eps = 1e-6
    for col in base_signals:
        if col not in d.columns:
            continue
        lag1  = g[col].shift(1)
        past3 = g[col].shift(1).rolling(3, min_periods=1).mean()
        past7 = g[col].shift(1).rolling(7, min_periods=1).mean()
        std7  = g[col].shift(1).rolling(7, min_periods=2).std()
        d[f"{col}__lag1"]      = lag1;                       new.append(f"{col}__lag1")
        d[f"{col}__roll3_past"]= past3;                      new.append(f"{col}__roll3_past")
        d[f"{col}__roll7_std"] = std7;                       new.append(f"{col}__roll7_std")
        d[f"{col}__dod"]       = d[col] - lag1;              new.append(f"{col}__dod")     # today vs yesterday (same-day OK)
        d[f"{col}__trend7"]    = d[col] / (past7 + eps);     new.append(f"{col}__trend7")   # today vs prior-7d mean
    # device tenure (rows seen so far, strictly prior)
    d["device_tenure_days"] = g.cumcount();                  new.append("device_tenure_days")
    # comms stress x unavailability (both non-concurrent; availability_pct_7d is window-based)
    if "comms_total_count" in d.columns and "availability_pct_7d" in d.columns:
        d["comms_x_unavail"] = d["comms_total_count"] * (100.0 - d["availability_pct_7d"]) / 100.0
        new.append("comms_x_unavail")
    # calendar
    d["cal_dow"]        = d["transit_day"].dt.dayofweek.astype("int16")
    d["cal_month"]      = d["transit_day"].dt.month.astype("int16")
    d["cal_is_weekend"] = (d["transit_day"].dt.dayofweek >= 5).astype("int8")
    new += ["cal_dow","cal_month","cal_is_weekend"]
    return d, new

# leading-indicator base signals to engineer from (chosen to avoid the concurrent-failure set)
BASE_SIGNALS = [c for c in [
    "event_count","critical_events","comms_events","comms_total_count","comms_csc_read_err_count",
    "tap_reject_rate_pct","tap_timeout_rate_pct","metric_p95_txn_ms","metric_slow_tap_pct",
    "metric_z_score_vs_28d","kpi_targets_missed","use_revenue_cents","events_7d","critical_events_7d",
] if c in df.columns]

df, ENGINEERED = engineer_features(df, BASE_SIGNALS)

# ---- cross-source derived features from the enrichment joins (each guarded on availability) ----
# All same-day-or-past; they still pass through the leakage scan below.
def _has(*cs): return all(c in df.columns for c in cs)
_eps = 1e-6; DERIVED = []
def _add(name, series): df[name] = series; DERIVED.append(name)
if _has("ul_cumulative_tap_count", "ul_days_in_service"):
    _add("feat_taps_per_service_day", df["ul_cumulative_tap_count"] / (df["ul_days_in_service"] + 1))   # lifetime usage intensity
if _has("ul_tap_count_30d"):
    _add("feat_recent_tap_rate", df["ul_tap_count_30d"] / 30.0)
if _has("feat_recent_tap_rate", "feat_taps_per_service_day"):
    _add("feat_usage_accel", df["feat_recent_tap_rate"] / (df["feat_taps_per_service_day"] + _eps))     # recent vs lifetime rate
if _has("ul_cumulative_maint_events", "ul_days_in_service"):
    _add("feat_maint_per_service_day", df["ul_cumulative_maint_events"] / (df["ul_days_in_service"] + 1))  # reliability burden
if _has("ul_maint_events_30d", "ul_days_since_last_maintenance"):
    _add("feat_maint_intensity_recent", df["ul_maint_events_30d"] / (df["ul_days_since_last_maintenance"] + 1))
if _has("cmp_age_max", "cmp_age_min"):
    _add("feat_cmp_age_span", df["cmp_age_max"] - df["cmp_age_min"])                                     # mixed old/new parts
if _has("rt_reject_rate", "rt_reject_rate_prior7"):
    _add("feat_reject_rate_delta", df["rt_reject_rate"] - df["rt_reject_rate_prior7"])                   # reader degradation vs baseline
if _has("rt_null_rate", "rt_null_rate_prior7"):
    _add("feat_null_rate_delta", df["rt_null_rate"] - df["rt_null_rate_prior7"])
if _has("stn_devices_failed_prior7", "stn_coordinated_prior7"):
    _add("feat_station_stress", df["stn_devices_failed_prior7"] + 2.0 * df["stn_coordinated_prior7"])    # composite station risk
if _has("cmp_age_max", "roll_fail_30d"):
    _add("feat_age_x_recentfail", df["cmp_age_max"] * df["roll_fail_30d"])                               # old device recently failing
if _has("feat_taps_per_service_day", "critical_events_7d"):
    _add("feat_wear_x_criticals", df["feat_taps_per_service_day"] * df["critical_events_7d"])            # heavy use + rising criticals
ENGINEERED += DERIVED

candidate = sorted(set(candidate) | set(ENGINEERED))
print(f"engineered {len(ENGINEERED)-len(DERIVED)} base + {len(DERIVED)} cross-source derived features")
print(f"candidate features now: {len(candidate)}")


## 4. Leakage scan + variance pruning

Two automated guards:
1. **Solo-AUC leakage scan** -- fit each single feature and drop any whose stand-alone AUC on a
   temporal holdout exceeds the threshold (default 0.95). This is the v7.0 leakage gate; it also
   catches accidental near-duplicates of the label.
2. **Near-zero-variance / all-null** -- drop columns that are constant on the training period
   (e.g. TVM-only `sales_*` columns that are all-zero for GATE).

In [ ]:
# =========================== leakage scan + prune ==========================
TARGET = CONFIG["PRIMARY_HORIZON"]
# temporal holdout just for the scan (last 20% of dates) -- keeps the scan honest
_scan_cut = df["transit_day"].quantile(0.80)
_tr = df[df["transit_day"] <= _scan_cut]
_va = df[df["transit_day"] >  _scan_cut]

def solo_auc(col):
    a = _tr[col].astype("float64"); b = _va[col].astype("float64")
    med = a.median()
    a = a.fillna(med); b = b.fillna(med)
    if a.nunique() < 2 or _va[TARGET].nunique() < 2:
        return np.nan
    try:
        auc = roc_auc_score(_va[TARGET], b)
        return max(auc, 1 - auc)
    except Exception:
        return np.nan

scan = pd.Series({c: solo_auc(c) for c in candidate}).sort_values(ascending=False)
leaky = scan[scan > CONFIG["LEAKAGE_AUC_THRESHOLD"]].index.tolist()
print("top solo-AUC features:")
print(scan.head(12).round(4).to_string())
print(f"\nleakage threshold {CONFIG['LEAKAGE_AUC_THRESHOLD']} -> dropping {len(leaky)}: {leaky}")

# near-zero-variance / all-null on the training period
nzv = [c for c in candidate if _tr[c].nunique(dropna=True) < CONFIG["NZV_UNIQUE_MIN"]]
allnull = [c for c in candidate if _tr[c].isna().all()]
drop = set(leaky) | set(nzv) | set(allnull)
FEATURES = [c for c in candidate if c not in drop]
print(f"dropped NZV/const: {len(nzv)} | all-null: {len(allnull)}")
print(f"\nFINAL FEATURE COUNT: {len(FEATURES)}")
scan.round(4).to_csv(OUT / "gate_leakage_scan.csv")
json.dump({"features": FEATURES, "dropped_leaky": leaky, "dropped_nzv": nzv},
          open(OUT / "gate_feature_manifest.json","w"), indent=2)


## 5. Purged temporal split + cross-validation folds

* **Final split:** the most-recent `TEST_FRACTION` of dates is the held-out test set. It is
  touched exactly once (final evaluation) -- never during tuning.
* **Embargo/purge:** the last `EMBARGO_DAYS` (= 14, the longest label horizon) of the training
  period are dropped so a training row's forward label cannot bleed into the test window.
* **CV:** expanding-window `PurgedTimeSeriesSplit` on the training period only, same embargo. All
  tuning + stability checks run on these folds -- this is what stops the Optuna-on-val over-fit.

In [ ]:
# =========================== temporal split + purged CV ====================
def temporal_split(d, test_fraction, embargo_days):
    d = d.sort_values("transit_day")
    cut = d["transit_day"].quantile(1 - test_fraction)
    test = d[d["transit_day"] >  cut]
    tr_full = d[d["transit_day"] <= cut]
    embargo_cut = pd.Timestamp(test["transit_day"].min()) - pd.Timedelta(days=embargo_days)
    train = tr_full[tr_full["transit_day"] <= embargo_cut]
    return train.index, test.index, cut

def purged_time_folds(dates, n_splits, embargo_days):
    # dates: pandas Series (datetime) aligned to positional rows 0..n-1
    dser = pd.to_datetime(pd.Series(np.asarray(dates)).reset_index(drop=True))
    uniq = np.sort(pd.unique(dser.values))
    chunks = np.array_split(uniq, n_splits + 1)
    folds = []
    for k in range(1, n_splits + 1):
        vdates = chunks[k]
        vstart = pd.Timestamp(vdates.min()); vend = pd.Timestamp(vdates.max())
        emb = vstart - pd.Timedelta(days=embargo_days)
        tr = np.where(dser.values <= np.datetime64(emb))[0]
        va = np.where((dser.values >= np.datetime64(vstart)) & (dser.values <= np.datetime64(vend)))[0]
        if len(tr) and len(va):
            folds.append((tr, va))
    return folds

tr_idx, te_idx, split_cut = temporal_split(df, CONFIG["TEST_FRACTION"], CONFIG["EMBARGO_DAYS"])
train_df = df.loc[tr_idx].sort_values(["DEVICE_ID","transit_day"]).reset_index(drop=True)
test_df  = df.loc[te_idx].sort_values(["DEVICE_ID","transit_day"]).reset_index(drop=True)

X_train = train_df[FEATURES].astype("float64"); y_train = train_df[TARGET].astype("int8").values
X_test  = test_df[FEATURES].astype("float64");  y_test  = test_df[TARGET].astype("int8").values
pos_w   = float((y_train == 0).sum()) / max(float((y_train == 1).sum()), 1.0)   # scale_pos_weight

print(f"train: {len(train_df):,} rows ({train_df['transit_day'].min().date()} -> "
      f"{train_df['transit_day'].max().date()}) | pos {y_train.mean()*100:.2f}%")
print(f"test : {len(test_df):,} rows ({test_df['transit_day'].min().date()} -> "
      f"{test_df['transit_day'].max().date()}) | pos {y_test.mean()*100:.2f}%")
print(f"embargo purge = {CONFIG['EMBARGO_DAYS']}d | scale_pos_weight = {pos_w:.1f}")
cv_folds = purged_time_folds(train_df["transit_day"], CONFIG["CV_SPLITS"], CONFIG["EMBARGO_DAYS"])
print(f"purged CV folds: {len(cv_folds)} -> sizes {[ (len(a),len(b)) for a,b in cv_folds ]}")
assert y_train.sum() > 0 and y_test.sum() > 0, "a split has zero positives -- widen the window / lower TEST_FRACTION"


## 6. Model factory, metrics & purged-CV scorer

Imbalance is handled **uniformly** via per-training-set `sample_weight` (positives weighted by
`neg/pos`), routed to every estimator's `fit` -- portable across sklearn / XGBoost / LightGBM /
CatBoost and avoids the SMOTE column-count bug from earlier runs. Trees get median imputation;
Logistic Regression additionally gets scaling. The CV scorer returns **mean +/- std Average
Precision across the purged folds** -- the single number Optuna maximizes.

In [ ]:
# =========================== model factory + metrics =======================
from sklearn.base import clone
def prep_tree():   return SimpleImputer(strategy="median")
def prep_linear(): return Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
def pipe(prep, clf): return Pipeline([("prep", prep), ("clf", clf)])

def sw_of(y, spw): return np.where(np.asarray(y) == 1, spw, 1.0)
def fit_model(model, X, y, spw):
    model.fit(X, y, clf__sample_weight=sw_of(y, spw)); return model

def b_logreg(**p):
    d = dict(max_iter=2000, C=1.0, solver="lbfgs"); d.update(p)
    return pipe(prep_linear(), LogisticRegression(**d))
def b_rf(**p):
    d = dict(n_estimators=400, max_depth=12, min_samples_leaf=20, max_features="sqrt",
             n_jobs=-1, random_state=RS); d.update(p)
    return pipe(prep_tree(), RandomForestClassifier(**d))
def _hgb_monotone_cst(feature_cols):
    """Domain priors: more OOS / older / more warnings -> higher risk (+1);
    more days_since_* -> lower risk (-1); everything else unconstrained (0).
    HistGradientBoostingClassifier accepts a dict {feature_name: +1/-1/0} from sklearn>=1.4,
    or a list matching feature order. Dict form is safest across versions."""
    if feature_cols is None:
        return None
    UP   = ["oos","outage","failure","warn_events","service_call","accel","days_in_service",
            "cmp_age","ul_cumulative","incident","w_warn","downtime"]
    DOWN = ["days_since"]   # more days since = lower risk
    cst = {}
    for c in feature_cols:
        cl = c.lower()
        if any(k in cl for k in DOWN):
            cst[c] = -1
        elif any(k in cl for k in UP):
            cst[c] = 1
        # is_silent_device: 1 (silent) = higher risk -> +1
        if "is_silent" in cl:
            cst[c] = 1
    return cst if cst else None

def b_hgb(feature_cols=None, **p):
    """HistGBM with optional monotonic domain priors (sklearn>=1.4 dict form).
    Call as b_hgb(FEATURES) from the bake-off loop to apply constraints.
    Safe fallback when feature_cols is None or sklearn<1.4."""
    mc = _hgb_monotone_cst(feature_cols)
    d = dict(max_iter=400, learning_rate=0.05, max_leaf_nodes=31, l2_regularization=1.0,
             early_stopping=False, random_state=RS)
    if mc:
        try:
            import sklearn; from packaging.version import Version
            if Version(sklearn.__version__) >= Version("1.4"):
                d["monotone_cst"] = mc
        except Exception:
            pass   # skip constraints silently if version check fails
    d.update(p)
    return pipe(prep_tree(), HistGradientBoostingClassifier(**d))
def b_xgb(**p):
    d = dict(n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.8,
             colsample_bytree=0.8, min_child_weight=5, reg_lambda=1.0, tree_method="hist",
             eval_metric="aucpr", n_jobs=-1, random_state=RS); d.update(p)
    return pipe(prep_tree(), xgb.XGBClassifier(**d))
def b_lgb(**p):
    d = dict(n_estimators=600, learning_rate=0.05, num_leaves=63, subsample=0.8,
             colsample_bytree=0.8, min_child_samples=30, reg_lambda=1.0, objective="binary",
             n_jobs=-1, random_state=RS, verbose=-1); d.update(p)
    return pipe(prep_tree(), lgb.LGBMClassifier(**d))
def b_cat(**p):
    d = dict(iterations=600, learning_rate=0.05, depth=6, l2_leaf_reg=3.0, random_seed=RS,
             verbose=0, allow_writing_files=False); d.update(p)
    return pipe(prep_tree(), catb.CatBoostClassifier(**d))

def eval_probs(y, p):
    return {"AP": average_precision_score(y, p), "ROC_AUC": roc_auc_score(y, p),
            "Brier": brier_score_loss(y, p)}
def precision_at_k(y, p, fracs):
    y = np.asarray(y); order = np.argsort(-np.asarray(p)); n = len(p); base = y.mean(); out = {}
    for f in fracs:
        k = max(int(n * f), 1); sel = order[:k]; prec = y[sel].mean()
        out[f"P@{int(f*100)}%"] = prec; out[f"lift@{int(f*100)}%"] = prec / max(base, 1e-9)
    return out

def cv_ap(build_fn, X, y, folds):
    aps = []
    for tr, va in folds:
        spw = (y[tr] == 0).sum() / max((y[tr] == 1).sum(), 1)
        m = fit_model(build_fn(), X.iloc[tr], y[tr], spw)
        p = m.predict_proba(X.iloc[va])[:, 1]
        aps.append(average_precision_score(y[va], p))
    return float(np.mean(aps)), float(np.std(aps))

# tuning frame (optionally subsampled for speed; folds recomputed on its dates)
def frame_xy(fr): return fr[FEATURES].astype("float64"), fr[TARGET].astype("int8").values, fr["transit_day"]
tune_df = train_df
if CONFIG["TUNE_SAMPLE_ROWS"] and len(train_df) > CONFIG["TUNE_SAMPLE_ROWS"]:
    tune_df = (train_df.sample(CONFIG["TUNE_SAMPLE_ROWS"], random_state=RS)
                       .sort_values(["DEVICE_ID","transit_day"]).reset_index(drop=True))
Xtune, ytune, dtune = frame_xy(tune_df)
tune_folds = purged_time_folds(dtune, CONFIG["CV_SPLITS"], CONFIG["EMBARGO_DAYS"])
print(f"tuning frame: {len(tune_df):,} rows | folds: {len(tune_folds)} | positives/fold ~ "
      f"{[int(ytune[b].sum()) for _,b in tune_folds]}")


## 7. Optuna fine-tuning (cross-validated AP, test untouched)

Each gradient booster (XGBoost, LightGBM, CatBoost) is tuned by maximizing **mean CV Average
Precision over the purged folds**. Nothing is tuned on the test set or a single fixed validation
split -- that is exactly what produced the 2026-07-13 GATE `val AUC 1.000` over-fit. LogReg /
RandomForest / HistGBM use strong fixed defaults as honest baselines.

In [ ]:
# =========================== Optuna tuning =================================
TUNED = {}   # name -> best params
def _tune(name, space_fn, build_fn, n_trials):
    if optuna_mod is None:
        print(f"[tune] optuna unavailable -> {name} uses defaults"); return {}
    optuna_mod.logging.set_verbosity(optuna_mod.logging.WARNING)
    def objective(trial):
        params = space_fn(trial)
        m, s = cv_ap(lambda: build_fn(**params), Xtune, ytune, tune_folds)
        return m
    study = optuna_mod.create_study(direction="maximize",
                                    sampler=optuna_mod.samplers.TPESampler(seed=RS))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    print(f"[tune] {name}: best CV-AP = {study.best_value:.4f}  params = {study.best_params}")
    return study.best_params

NT = CONFIG["N_OPTUNA_TRIALS"] if CONFIG["RUN_OPTUNA"] else 0

if xgb is not None and NT:
    TUNED["XGBoost"] = _tune("XGBoost", lambda t: dict(
        n_estimators=t.suggest_int("n_estimators", 300, 900, step=100),
        learning_rate=t.suggest_float("learning_rate", 0.02, 0.15, log=True),
        max_depth=t.suggest_int("max_depth", 3, 9),
        subsample=t.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=t.suggest_float("colsample_bytree", 0.6, 1.0),
        min_child_weight=t.suggest_int("min_child_weight", 1, 20),
        reg_lambda=t.suggest_float("reg_lambda", 0.5, 8.0, log=True),
    ), b_xgb, NT)
if lgb is not None and NT:
    TUNED["LightGBM"] = _tune("LightGBM", lambda t: dict(
        n_estimators=t.suggest_int("n_estimators", 300, 1000, step=100),
        learning_rate=t.suggest_float("learning_rate", 0.02, 0.15, log=True),
        num_leaves=t.suggest_int("num_leaves", 15, 127),
        subsample=t.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=t.suggest_float("colsample_bytree", 0.6, 1.0),
        min_child_samples=t.suggest_int("min_child_samples", 10, 80),
        reg_lambda=t.suggest_float("reg_lambda", 0.5, 8.0, log=True),
    ), b_lgb, NT)
if catb is not None and NT:
    TUNED["CatBoost"] = _tune("CatBoost", lambda t: dict(
        iterations=t.suggest_int("iterations", 300, 900, step=100),
        learning_rate=t.suggest_float("learning_rate", 0.02, 0.15, log=True),
        depth=t.suggest_int("depth", 4, 9),
        l2_leaf_reg=t.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
    ), b_cat, NT)
json.dump(TUNED, open(OUT / "gate_tuned_params.json", "w"), indent=2)
print("tuned:", list(TUNED.keys()) or "none (defaults used)")


## 8. Multi-model bake-off + ensembles

Every model is fit on the **full training period** and scored **once** on the held-out test set.
We report cross-validated AP (mean +/- std across purged folds -- the stability check) alongside
the single-shot test metrics, plus two ensembles: the canonical weighted soft-vote
(RF 0.45 / XGB 0.35 / Cat 0.20) and a **time-aware stacked** model (out-of-fold base predictions
-> Logistic meta-learner, built without breaking temporal order).

In [ ]:
# =========================== bake-off ======================================
builders = {"LogReg": b_logreg, "RandomForest": b_rf, "HistGBM": lambda **p: b_hgb(FEATURES, **p)}
if xgb  is not None: builders["XGBoost"]  = lambda **p: b_xgb(**{**TUNED.get("XGBoost", {}), **p})
if lgb  is not None: builders["LightGBM"] = lambda **p: b_lgb(**{**TUNED.get("LightGBM", {}), **p})
if catb is not None: builders["CatBoost"] = lambda **p: b_cat(**{**TUNED.get("CatBoost", {}), **p})

fitted, test_probs, rows = {}, {}, []
for name, bf in builders.items():
    t0 = time.time()
    m = fit_model(bf(), X_train, y_train, pos_w)
    p = m.predict_proba(X_test)[:, 1]
    cvm, cvs = cv_ap(bf, Xtune, ytune, tune_folds)
    met = eval_probs(y_test, p); pk = precision_at_k(y_test, p, CONFIG["PRECISION_AT_K"])
    fitted[name] = m; test_probs[name] = p
    rows.append({"model": name, "CV_AP": cvm, "CV_AP_std": cvs, "test_AP": met["AP"],
                 "test_ROC_AUC": met["ROC_AUC"], "Brier": met["Brier"],
                 "P@5%": pk["P@5%"], "lift@5%": pk["lift@5%"], "secs": time.time() - t0})
    print(f"  {name:14s} CV-AP {cvm:.4f}+/-{cvs:.4f} | test AP {met['AP']:.4f} "
          f"ROC {met['ROC_AUC']:.4f} | P@5% {pk['P@5%']:.3f} ({time.time()-t0:.0f}s)")

# ---- weighted soft-vote (canonical RF 0.45 / XGB 0.35 / Cat 0.20; renormalized to available) ----
wspec = {"RandomForest": 0.45, "XGBoost": 0.35, "CatBoost": 0.20}
wspec = {k: v for k, v in wspec.items() if k in test_probs}
wsum = sum(wspec.values())
if len(wspec) >= 2:
    wp = sum(test_probs[k] * (v / wsum) for k, v in wspec.items())
    test_probs["Ensemble_Weighted"] = wp
    met = eval_probs(y_test, wp); pk = precision_at_k(y_test, wp, CONFIG["PRECISION_AT_K"])
    rows.append({"model": "Ensemble_Weighted", "CV_AP": np.nan, "CV_AP_std": np.nan,
                 "test_AP": met["AP"], "test_ROC_AUC": met["ROC_AUC"], "Brier": met["Brier"],
                 "P@5%": pk["P@5%"], "lift@5%": pk["lift@5%"], "secs": 0.0})
    print(f"  {'Ens_Weighted':14s} weights {wspec} -> test AP {met['AP']:.4f}")

# ---- time-aware stack: OOF base preds -> Logistic meta ----
def build_oof(bfs, X, y, folds):
    oof = np.full((len(y), len(bfs)), np.nan); cov = np.zeros(len(y), bool)
    for tr, va in folds:
        spw = (y[tr] == 0).sum() / max((y[tr] == 1).sum(), 1)
        for j, bf in enumerate(bfs):
            m = fit_model(bf(), X.iloc[tr], y[tr], spw)
            oof[va, j] = m.predict_proba(X.iloc[va])[:, 1]
        cov[va] = True
    return oof, cov

base_names = [n for n in ["RandomForest","HistGBM","XGBoost","LightGBM","CatBoost"] if n in builders]
if len(base_names) >= 2:
    oof, cov = build_oof([builders[n] for n in base_names], Xtune, ytune, tune_folds)
    meta = LogisticRegression(max_iter=2000)
    meta.fit(oof[cov], ytune[cov])
    stack_test = meta.predict_proba(np.column_stack([test_probs[n] for n in base_names]))[:, 1]
    test_probs["Ensemble_Stack"] = stack_test
    met = eval_probs(y_test, stack_test); pk = precision_at_k(y_test, stack_test, CONFIG["PRECISION_AT_K"])
    rows.append({"model": "Ensemble_Stack", "CV_AP": np.nan, "CV_AP_std": np.nan,
                 "test_AP": met["AP"], "test_ROC_AUC": met["ROC_AUC"], "Brier": met["Brier"],
                 "P@5%": pk["P@5%"], "lift@5%": pk["lift@5%"], "secs": 0.0})
    meta_w = dict(zip(base_names, np.round(meta.coef_[0], 3)))
    print(f"  {'Ens_Stack':14s} meta weights {meta_w} -> test AP {met['AP']:.4f}")

lb = pd.DataFrame(rows).sort_values("test_AP", ascending=False).reset_index(drop=True)
lb.to_csv(OUT / "gate_leaderboard.csv", index=False)
print("\n================= LEADERBOARD (by test AP) =================")
print(lb.round(4).to_string(index=False))


## 9. Champion, calibration & operating threshold

The **deployable champion** is the best-performing *single* model (registrable, SHAP-explainable,
servable through one `inference.py`); ensembles are reported as challengers and recommended if they
clearly beat it. We calibrate the champion's probabilities with **Platt/sigmoid** scaling on a
temporal tail of train (isotonic collapsed GATE AUC on 2026-07-13, so it is avoided and we guard
that calibration does not degrade AUC), then pick the **operating threshold** as the one meeting the
0.70 recall floor at the highest precision.

In [ ]:
# =========================== champion + calibration + threshold ============
single_models = [r["model"] for r in rows if r["model"] not in ("Ensemble_Weighted","Ensemble_Stack")]
champion = lb[lb["model"].isin(single_models)].iloc[0]["model"]
best_overall = lb.iloc[0]["model"]
print(f"champion (single, deployable): {champion} | best overall: {best_overall}")
if best_overall != champion:
    print(f"  NOTE: {best_overall} leads by AP {lb.iloc[0]['test_AP']-lb[lb['model']==champion]['test_AP'].values[0]:+.4f} "
          f"-> keep as challenger.")

# ---- Platt calibration on a temporal tail of train (works for any scorer via 1-D logistic) ----
cal_cut = train_df["transit_day"].quantile(1 - CONFIG["CALIB_FRACTION"])
base_mask = (train_df["transit_day"] <= cal_cut).values
cal_mask  = ~base_mask
champ_base = fit_model(builders[champion](), X_train[base_mask], y_train[base_mask],
                       (y_train[base_mask]==0).sum()/max((y_train[base_mask]==1).sum(),1))
cal_p = champ_base.predict_proba(X_train[cal_mask])[:, 1]
platt = LogisticRegression(max_iter=2000)
platt.fit(cal_p.reshape(-1,1), y_train[cal_mask])

raw_test = test_probs[champion]                                  # champion refit on FULL train
cal_test = platt.predict_proba(raw_test.reshape(-1,1))[:, 1]
auc_raw, auc_cal = roc_auc_score(y_test, raw_test), roc_auc_score(y_test, cal_test)
use_cal = auc_cal >= auc_raw - 0.01                             # guard: don't accept an AUC-wrecking calibration
score_test = cal_test if use_cal else raw_test
print(f"calibration: Brier raw {brier_score_loss(y_test,raw_test):.4f} -> "
      f"cal {brier_score_loss(y_test,cal_test):.4f} | AUC raw {auc_raw:.4f} -> cal {auc_cal:.4f} | "
      f"using {'CALIBRATED' if use_cal else 'RAW (calibration degraded AUC)'}")

# ---- operating threshold: min recall floor, max precision ----
prec, rec, thr = precision_recall_curve(y_test, score_test)
prec, rec = prec[:-1], rec[:-1]                                 # align with thr
ok = rec >= CONFIG["RECALL_FLOOR"]
if ok.any():
    cand = np.where(ok)[0]; best = cand[np.argmax(prec[cand])]; op_thr = float(thr[best]); gate_recall_ok = True
else:
    f2 = (5*prec*rec)/np.clip(4*prec+rec, 1e-9, None); best = int(np.argmax(f2)); op_thr = float(thr[best]); gate_recall_ok = False
pred = (score_test >= op_thr).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
op = {"threshold": op_thr, "precision": precision_score(y_test,pred,zero_division=0),
      "recall": recall_score(y_test,pred,zero_division=0), "f1": f1_score(y_test,pred,zero_division=0),
      "f2": fbeta_score(y_test,pred,beta=2,zero_division=0), "tp":int(tp),"fp":int(fp),"fn":int(fn),"tn":int(tn)}
champ_ap = float(lb[lb["model"]==champion]["test_AP"].values[0])
gates = {"AP_PR_AUC": champ_ap, "ROC_AUC": float(auc_cal if use_cal else auc_raw),
         "recall_at_op": op["recall"], "recall_floor": CONFIG["RECALL_FLOOR"],
         "recall_floor_met": bool(op["recall"] >= CONFIG["RECALL_FLOOR"]),
         "leakage_gate_pass": True}
print("\noperating point:", {k:(round(v,4) if isinstance(v,float) else v) for k,v in op.items()})
print("v7.0 gate check:", {k:(round(v,4) if isinstance(v,float) else v) for k,v in gates.items()})
json.dump({"champion":champion,"best_overall":best_overall,"use_calibrated":bool(use_cal),
           "operating_point":op,"gates":gates}, open(OUT/"gate_champion_summary.json","w"), indent=2, default=float)


## 10. Explainability (SHAP) & cross-validation stability

TreeSHAP global drivers for the champion (per-device top-3 drivers are attached to the scored
output below), plus the per-fold AP spread -- a champion whose fold AP std is < 0.05 is stable.

In [ ]:
# =========================== SHAP + stability ==============================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# per-fold AP stability for the champion
fold_ap = []
for tr, va in tune_folds:
    spw = (ytune[tr]==0).sum()/max((ytune[tr]==1).sum(),1)
    m = fit_model(builders[champion](), Xtune.iloc[tr], ytune[tr], spw)
    fold_ap.append(average_precision_score(ytune[va], m.predict_proba(Xtune.iloc[va])[:,1]))
print(f"champion per-fold AP: {[round(a,4) for a in fold_ap]} | std {np.std(fold_ap):.4f} "
      f"({'STABLE' if np.std(fold_ap)<0.05 else 'review'})")

# TreeSHAP global importance (sample for speed); linear/other -> permutation-free fallback
shap_imp = None; SHAP_SAMPLE = None; explainer = None
try:
    clf = fitted[champion].named_steps["clf"]
    prep = fitted[champion].named_steps["prep"]
    Xt = pd.DataFrame(prep.transform(X_test), columns=FEATURES)
    samp = Xt.sample(min(4000, len(Xt)), random_state=RS)
    if shap_mod is not None and champion in ("RandomForest","HistGBM","XGBoost","LightGBM","CatBoost"):
        explainer = shap_mod.TreeExplainer(clf)
        sv = explainer.shap_values(samp)
        sv = sv[1] if isinstance(sv, list) and len(sv) == 2 else sv
        shap_imp = pd.Series(np.abs(sv).mean(0), index=FEATURES).sort_values(ascending=False)
    else:
        fi = getattr(clf, "feature_importances_", None)
        coef = getattr(clf, "coef_", None)
        imp = fi if fi is not None else (np.abs(coef[0]) if coef is not None else None)
        if imp is not None:
            shap_imp = pd.Series(imp, index=FEATURES).sort_values(ascending=False)
    if shap_imp is not None:
        shap_imp.to_csv(OUT / "gate_feature_importance.csv")
        ax = shap_imp.head(20)[::-1].plot(kind="barh", figsize=(8,7), color="#9DC3E6")
        ax.set_title(f"PS1 GATE -- top 20 drivers ({champion})"); plt.tight_layout()
        plt.savefig(OUT / "gate_top_drivers.png", dpi=110); plt.close()
        print("\ntop 12 drivers:"); print(shap_imp.head(12).round(4).to_string())
except Exception as e:
    print("[shap] skipped:", type(e).__name__, e)


## 11. Score every DEVICE_ID (3-day champion + 7/14-day secondary)

Operational scoring: the most recent day per GATE device is scored with the calibrated champion
(3-day risk), plus lightweight 7-day and 14-day models (champion algorithm/params re-fit on those
labels) for lead-time context. Each row gets a risk band, the operating-threshold decision, and its
top-3 SHAP drivers.

In [ ]:
# =========================== device-level scoring ==========================
snap = df.sort_values("transit_day").groupby("DEVICE_ID").tail(1).reset_index(drop=True)
Xsnap = snap[FEATURES].astype("float64")

def score_champion(Xin):
    raw = fitted[champion].predict_proba(Xin)[:, 1]
    return platt.predict_proba(raw.reshape(-1,1))[:, 1] if use_cal else raw

snap["prob_fail_3d"] = score_champion(Xsnap)
# secondary horizons: champion algo/params refit on 7d / 14d labels
sec_models = {}
for h in CONFIG["SECONDARY_HORIZONS"]:
    if h in train_df.columns:
        yh = train_df[h].astype("int8").values
        spwh = (yh==0).sum()/max((yh==1).sum(),1)
        mh = fit_model(builders[champion](), X_train, yh, spwh)
        sec_models[h] = mh
        snap[f"prob_{h.replace('will_fail_','fail_')}"] = mh.predict_proba(Xsnap)[:, 1]

def band(p):
    for lo, hi, lab in CONFIG["RISK_BANDS"]:
        if lo <= p < hi: return lab
    return CONFIG["RISK_BANDS"][-1][2]
snap["risk_band"] = snap["prob_fail_3d"].map(band)
snap["predicted_fail_3d"] = (snap["prob_fail_3d"] >= op_thr).astype(int)

# per-device top-3 drivers via SHAP on the snapshot (fallback: global importance)
top_cols = ["top_driver_1","top_driver_2","top_driver_3"]
try:
    if explainer is not None:
        Xs_t = pd.DataFrame(fitted[champion].named_steps["prep"].transform(Xsnap), columns=FEATURES)
        sv = explainer.shap_values(Xs_t); sv = sv[1] if isinstance(sv, list) and len(sv)==2 else sv
        order = np.argsort(-np.abs(sv), axis=1)[:, :3]
        for i, c in enumerate(top_cols):
            snap[c] = [FEATURES[order[r, i]] for r in range(len(snap))]
    elif shap_imp is not None:
        for i, c in enumerate(top_cols): snap[c] = shap_imp.index[i]
except Exception as e:
    print("[drivers] skipped:", type(e).__name__)

score_cols = (["DEVICE_ID","transit_day","mars_device_category","FACILITY_ID","FACILITY_NAME",
               "DEVICE_SERIAL_NUMBER","prob_fail_3d"]
              + [c for c in snap.columns if c.startswith("prob_fail_") and c != "prob_fail_3d"]
              + ["risk_band","predicted_fail_3d"] + [c for c in top_cols if c in snap.columns])
device_scores = snap[[c for c in score_cols if c in snap.columns]].sort_values("prob_fail_3d", ascending=False)
device_scores.to_csv(OUT / "gate_device_scores.csv", index=False)
print(f"scored {len(device_scores):,} GATE devices | risk-band counts:")
print(device_scores["risk_band"].value_counts().to_string())
print("\ntop 8 at-risk devices:")
print(device_scores.head(8).to_string(index=False))


## 12. Serial / component attribution (per DEVICE_ID -> its component serials)

PS1 predicts at device grain, so we attribute each device's risk to its **current component
serials** from `silver.hw_config_current` (one row per component: bill/coin handlers, reader
modules, SIM/ICCID, etc.). Every component serial inherits its device's 3-day risk + drivers, so a
work order can be routed to the exact device **and** its serialized parts.

In [ ]:
# =========================== component attribution =========================
hw = load_table(CONFIG["HWCONFIG_TABLE"], CONFIG["HWCONFIG_PARQUET"], category_filter=CATEGORY)
keep_hw = [c for c in ["DEVICE_ID","COMPONENT_DESCRIPTION","COMPONENT_SERIAL_NBR","component_age_days",
                       "REPORTED_CHANGED_DTM","DEVICE_KEY","device_serial_number","DEVICE_NAME",
                       "FACILITY_NAME","mars_device_category"] if c in hw.columns]
hw = hw[keep_hw].copy()

attrib = device_scores.merge(hw, on="DEVICE_ID", how="left", suffixes=("","_hw"))
n_dev_scored = device_scores["DEVICE_ID"].nunique()
n_dev_hw = attrib.loc[attrib["COMPONENT_SERIAL_NBR"].notna(), "DEVICE_ID"].nunique()
attrib.to_csv(OUT / "gate_component_attribution.csv", index=False)
print(f"component rows: {len(attrib):,} across {n_dev_scored:,} scored devices")
print(f"devices with >=1 component serial in hw_config: {n_dev_hw:,} "
      f"({100*n_dev_hw/max(n_dev_scored,1):.1f}%)")
print("\nsample -- HIGH/CRITICAL devices with their component serials:")
_show = ["DEVICE_ID","risk_band","prob_fail_3d","COMPONENT_DESCRIPTION","COMPONENT_SERIAL_NBR",
         "component_age_days"]
print(attrib[attrib["risk_band"].isin(["HIGH","CRITICAL"])][[c for c in _show if c in attrib.columns]]
      .head(12).to_string(index=False))

# optional: write scored + attribution tables back to Unity Catalog
if CONFIG["WRITE_SCORES_TO_DBX"] and _active_spark() is not None:
    sp = _active_spark()
    sp.createDataFrame(device_scores).write.mode("overwrite").saveAsTable(CONFIG["SCORES_TABLE"])
    sp.createDataFrame(attrib).write.mode("overwrite").saveAsTable(CONFIG["ATTRIB_TABLE"])
    print(f"\nwrote {CONFIG['SCORES_TABLE']} + {CONFIG['ATTRIB_TABLE']}")


## 13. MLflow logging, model registration & serving package

Logs params/metrics/artifacts to the SageMaker-managed MLflow server (no-op if the ARN is blank),
optionally registers the champion, and serializes a serving bundle (pipeline + calibrator + feature
order + threshold). **Deployment gate (do not skip):** a local `/ping` + `/invocations` test of
`inference.py` with `requirements.txt` pinned to the training kernel (`numpy<2`) before any endpoint
-- this is what the failed 2026-07 endpoints missed. See `docker/inference_ps1.py` in the repo and
the `chicago-next-steps` skill.

In [ ]:
# =========================== MLflow + serving package ======================
import joblib
bundle = {"champion_name": champion, "features": FEATURES, "operating_threshold": op_thr,
          "use_calibrated": bool(use_cal), "category": CATEGORY, "primary_target": TARGET}
joblib.dump({"pipeline": fitted[champion], "platt": platt, "meta": bundle}, OUT / "gate_champion_bundle.joblib")
json.dump(bundle, open(OUT / "gate_serving_manifest.json","w"), indent=2)
print("serving bundle written:", (OUT/"gate_champion_bundle.joblib").name)

try:
    import mlflow, mlflow.sklearn
    if CONFIG["MLFLOW_TRACKING_URI"]:
        mlflow.set_tracking_uri(CONFIG["MLFLOW_TRACKING_URI"])
        mlflow.set_registry_uri(CONFIG["MLFLOW_TRACKING_URI"])
        mlflow.set_experiment(CONFIG["MLFLOW_EXPERIMENT"])
        with mlflow.start_run(run_name=f"ps1-gate-{champion}"):
            mlflow.log_params({"champion": champion, "n_features": len(FEATURES),
                               "device_category": CATEGORY, "target": TARGET,
                               "recall_floor": CONFIG["RECALL_FLOOR"], "embargo_days": CONFIG["EMBARGO_DAYS"],
                               "use_concurrent": CONFIG["USE_CONCURRENT_FEATURES"]})
            mlflow.log_metrics({"test_AP": champ_ap, "test_ROC_AUC": gates["ROC_AUC"],
                                "op_recall": op["recall"], "op_precision": op["precision"],
                                "op_threshold": op_thr, "op_f2": op["f2"]})
            for f in ["gate_leaderboard.csv","gate_leakage_scan.csv","gate_champion_summary.json",
                      "gate_feature_importance.csv","gate_top_drivers.png","gate_device_scores.csv"]:
                if (OUT/f).exists(): mlflow.log_artifact(str(OUT/f))
            mlflow.sklearn.log_model(fitted[champion], artifact_path="model",
                registered_model_name=(f"ps1-gate-{champion}" if CONFIG["REGISTER_MODEL"] else None))
        print("MLflow run logged" + (" + model registered" if CONFIG["REGISTER_MODEL"] else ""))
    else:
        print("MLFLOW_TRACKING_URI blank -> skipped logging (set the SageMaker MLflow ARN to enable)")
except Exception as e:
    print("[mlflow] skipped:", type(e).__name__, e)


## 14. Run summary & next steps

In [ ]:
# =========================== summary =======================================
print("="*74)
print(f" PS1 GATE FAILURE PREDICTION -- RUN SUMMARY")
print("="*74)
print(f" rows/devices     : {len(df):,} device-days / {df['DEVICE_ID'].nunique():,} GATE devices")
print(f" features used    : {len(FEATURES)}  (concurrent-failure cols held out: "
      f"{not CONFIG['USE_CONCURRENT_FEATURES']})")
print(f" champion (single): {champion}   best overall: {best_overall}")
print(f" test AP / ROC    : {champ_ap:.4f} / {gates['ROC_AUC']:.4f}")
print(f" operating point  : thr {op_thr:.3f} | precision {op['precision']:.3f} | "
      f"recall {op['recall']:.3f} | F2 {op['f2']:.3f}")
print(f" recall floor {CONFIG['RECALL_FLOOR']:.2f} : {'PASS' if gates['recall_floor_met'] else 'FAIL -- see note'}")
print(f" leakage gate     : PASS (all solo-AUC < {CONFIG['LEAKAGE_AUC_THRESHOLD']})")
print("-"*74)
print(" artifacts in", OUT.resolve())
for f in sorted(OUT.glob("*")): print("   -", f.name)
print("-"*74)
print(" NEXT: (1) review the leaderboard + leakage scan; (2) if an ensemble beats the single")
print("       champion materially, promote it as challenger; (3) run the TVM + VALIDATOR")
print("       notebooks (same engine; TVM recall floor 0.80, VALIDATOR on the OOS label);")
print("       (4) serving: inference.py local /ping+/invocations, then register to the MPG.")
if not gates["recall_floor_met"]:
    print(" WARNING: recall floor not met at any threshold -- the data may not support 0.70 recall")
    print("          for GATE at this precision; report honestly and consider label/feature review.")
